In [ ]:
import numpy as np
import pandas as pd
import random
from functools import partial

from collections import defaultdict
from sklearn import metrics

import src.load_data as ld
import src.set_analysis_func as func
import src.func_optimized as func_opt


In [2]:
# load embedding
node_vectors = np.loadtxt(
    './data/embedding/node2vec_consensus.csv', delimiter=',')
node_list = []
with open('./data/embedding/consensus_node.txt', 'r') as f:
    for line in f:
        node_list.append(line.strip())
        
S = metrics.pairwise.cosine_similarity(node_vectors, node_vectors)

In [3]:
# create gene to embedding id mapping
g_node2index = {j:i for i,j in enumerate(node_list)}
g_index2node = {i:j for i,j in enumerate(node_list)}
g_node2index = defaultdict(lambda:-1, g_node2index)

In [4]:
# load gene set data
GO_data = ld.load_gmt(
    './data/gene_sets/hsa_experimental_eval_BP_propagated.gmt')

GO2indices = ld.term2indexes(
    GO_data, g_node2index, upper=300, lower=10)

In [5]:
# generate background gene list
GO_all_genes = set()
for x in GO_data:
    GO_all_genes = GO_all_genes.union(GO_data[x])
    
GO_all_genes = GO_all_genes.intersection(node_list)
GO_all_indices = [g_node2index[x] for x in GO_all_genes]

In [6]:
def sample_term_pairs(term_list, n_pairs, seed=0):
    """
    Sample n_pairs random ordered term pairs (t1, t2) with t1 != t2
    from term_list, with a fixed seed for reproducibility.
    """
    rng = random.Random(seed)
    pairs = []
    m = len(term_list)
    for _ in range(n_pairs):
        t1, t2 = rng.sample(term_list, 2)
        pairs.append((t1, t2))
    return pairs

GO_terms = list(GO2indices.keys())
term_pairs = sample_term_pairs(GO_terms, n_pairs=100, seed=42)
len(term_pairs), term_pairs[0]


(100, ('GO:0010765', 'GO:0010876'))

In [7]:
import time
import statistics as stats

def benchmark_andes_callable(andes_callable, term_pairs, warmup=5):
    """
    andes_callable: something like f(terms) -> (true_score, z_score)
    term_pairs: list of (term1, term2)
    warmup: number of calls to ignore for warmup
    """
    # warmup to trigger any lazy imports / cache effects
    for t in term_pairs[:warmup]:
        _ = andes_callable(t)

    times = []

    for t in term_pairs:
        start = time.perf_counter()
        _ = andes_callable(t)
        end = time.perf_counter()
        times.append(end - start)

    total = sum(times)
    avg = total / len(times)
    median = stats.median(times)

    return {
        "n_calls": len(term_pairs),
        "total_seconds": total,
        "avg_seconds": avg,
        "median_seconds": median,
    }


In [12]:
def warmup_numba():
    """
    Call this once at startup to trigger numba compilation.
    Avoids compilation overhead on first real call.
    """
    from src.func_optimized import (
        _compute_bma_from_indices,
        _andes_core_cached,
        _andes_background_parallel,
    )
    # Small dummy problem
    dummy_matrix = np.random.randn(50, 50).astype(np.float64)
    dummy_idx = np.arange(10, dtype=np.int64)
    dummy_pop = np.arange(50, dtype=np.int64)

    # Trigger helpers
    _compute_bma_from_indices(dummy_matrix, dummy_idx, dummy_idx)

    # Build tiny cached-style random index arrays for warmup
    ite = 5
    dummy_rand1 = np.stack([dummy_idx for _ in range(ite)], axis=0)
    dummy_rand2 = np.stack([dummy_idx for _ in range(ite)], axis=0)
    dummy_rand1 = np.ascontiguousarray(dummy_rand1, dtype=np.int64)
    dummy_rand2 = np.ascontiguousarray(dummy_rand2, dtype=np.int64)

    _andes_core_cached(dummy_matrix, dummy_idx, dummy_idx, dummy_rand1, dummy_rand2)

    # If you also use the older background kernels, you can still warm them:
    _andes_background_parallel(dummy_matrix, 10, 10, dummy_pop, dummy_pop, 10, 42)

    print("Numba compilation complete.")


In [13]:
warmup_numba()

Numba compilation complete.


In [14]:

f_old = partial(
    func.andes,
    matrix=S,
    g1_term2index=GO2indices,
    g2_term2index=GO2indices,
    g1_population=GO_all_indices,
    g2_population=GO_all_indices,
)

f_new = partial(
    func_opt.andes,
    matrix=S,
    g1_term2index=GO2indices,
    g2_term2index=GO2indices,
    g1_population=GO_all_indices,
    g2_population=GO_all_indices,
)


f_cached = partial(
    func_opt.andes_cached,
    matrix=S,
    g1_term2index=GO2indices,
    g2_term2index=GO2indices,
    g1_population=GO_all_indices,
    g2_population=GO_all_indices,
)

f_np_cached = partial(
    func_opt.andes_cached_np,
    matrix=S,
    g1_term2index=GO2indices,
    g2_term2index=GO2indices,
    g1_population=GO_all_indices,
    g2_population=GO_all_indices,
)

f_numba = partial(
    func_opt.andes_numba_parallel,
    matrix=S,
    g1_term2index=GO2indices,
    g2_term2index=GO2indices,
    g1_population=GO_all_indices,
    g2_population=GO_all_indices,
    seed=123,      # so it’s reproducible
)

f_final = partial(
    func_opt.andes_numba_cached,
    matrix=S,
    g1_term2index=GO2indices,
    g2_term2index=GO2indices,
    g1_population=GO_all_indices,
    g2_population=GO_all_indices,
    seed=123,      # so it’s reproducible
)

In [15]:
results_old = benchmark_andes_callable(f_old, term_pairs)
results_new = benchmark_andes_callable(f_new, term_pairs)
results_cached = benchmark_andes_callable(f_cached, term_pairs)
# results_np_cached = benchmark_andes_callable(f_np_cached, term_pairs)
results_numba = benchmark_andes_callable(f_numba, term_pairs)
results_final = benchmark_andes_callable(f_final, term_pairs)

In [16]:
results_old, results_new, results_cached, results_numba, results_final

({'n_calls': 100,
  'total_seconds': 4.662793167633936,
  'avg_seconds': 0.04662793167633936,
  'median_seconds': 0.032940958510152996},
 {'n_calls': 100,
  'total_seconds': 4.0677975048311055,
  'avg_seconds': 0.040677975048311055,
  'median_seconds': 0.026181645516771823},
 {'n_calls': 100,
  'total_seconds': 3.9979586654808372,
  'avg_seconds': 0.03997958665480837,
  'median_seconds': 0.02565249992767349},
 {'n_calls': 100,
  'total_seconds': 1.951311250566505,
  'avg_seconds': 0.01951311250566505,
  'median_seconds': 0.01900697947712615},
 {'n_calls': 100,
  'total_seconds': 0.8440251187421381,
  'avg_seconds': 0.008440251187421382,
  'median_seconds': 0.007737124979030341})